# Furiosa NPU · Holter 추론 준비

서버에서 실행합니다. 현재 서버에는 `/dev/fuse`가 없어 SSHFS 마운트가 구성되지 않았습니다. 관리자가 안전한 읽기 전용 마운트를 제공한 후 아래 경로를 설정하세요. 이 노트북은 모델 예측이나 의료 판정을 수행하지 않습니다.


In [ ]:
import subprocess
from pathlib import Path

import numpy as np
import torch  # torch를 먼저 import해야 설치된 furiosa.torch 자동 로딩이 정상 동작

DATA_DIR = Path("/path/to/read-only/Holter/windows")  # 서버의 실제 마운트 경로로 변경
print("torch:", torch.__version__)
print("NPU available:", torch.rngd.is_available())
subprocess.run(["furiosa-smi", "info"], check=True)

## NPU 연산 확인

실제 NPU에서 연산한 값을 CPU 결과와 비교합니다. 이 검사는 모델 호환성을 보장하지 않습니다.


In [ ]:
assert torch.rngd.is_available(), "Furiosa NPU를 사용할 수 없습니다."
x = torch.tensor([1.0, 2.0])
actual = (x.to("rngd") + x.to("rngd")).cpu()
torch.testing.assert_close(actual, x + x)
print("NPU 연산 확인 완료")

## 데이터 형식 확인

원본은 읽기만 합니다. 전체 파일을 메모리에 올리지 않도록 메모리 매핑을 사용합니다. 환자 식별자와 파형 값은 출력하지 않습니다.


In [ ]:
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"데이터 마운트를 확인하고 DATA_DIR을 설정하세요: {DATA_DIR}")
files = sorted(DATA_DIR.glob("*.npy"))
if not files:
    raise FileNotFoundError("마운트 경로에 NPY 파일이 없습니다.")
array = np.load(files[0], mmap_mode="r", allow_pickle=False)
print("NPY 파일 수:", len(files))
print("첫 파일 shape:", array.shape, "dtype:", array.dtype)
if array.ndim != 2 or array.shape[1] != 3:
    raise ValueError("예상한 (samples, 3) 형식이 아닙니다. 모델 입력 계약을 확인하세요.")
chunk = np.asarray(array[:6000], dtype=np.float32)
print("검사용 구간 shape:", chunk.shape, "finite:", bool(np.isfinite(chunk).all()))

## 모델 연결 전 확인

선택한 체크포인트와 모델 정의, sampling rate, 구간 길이, 정규화, 채널 순서, 출력 해석을 확인해야 합니다. CPU 기준 출력과 NPU 출력을 비교하고 실제 모델의 연산자 호환성을 검증한 뒤 추론을 연결하세요. 전체 데이터는 네트워크 연결이 유지되는 동안 접근할 수 있습니다.
